# Per-factor shape scaling (c_A, c_B) — does asymmetric radius beat equal-ρ?

Protagonist (`diag-shampoo-polar-lora`) with a per-factor shape coefficient folded into the operator-norm radius:

$$c_A = (r/d_{in})^a,\quad c_B = (d_{out}/r)^b,\qquad \rho = \frac{\eta}{c_A\,\sigma_{\max}(B) + c_B\,\sigma_{\max}(A)},\quad \sigma_{\max}(\dot A)=c_A\rho,\ \sigma_{\max}(\dot B)=c_B\rho.$$

$(a,b)=(0,0)$ is bit-identical to the shipped protagonist (equal radius). Cell: **Llama-3.2-1B / openmath / r=64** — the rank where the output-contribution imbalance was largest (δ²/δ¹≈0.54 on chord-tight), i.e. the most sensitive test.

**Arms** (a × b grid): (0,0) baseline, (0,½) Keller, (¼,0) A-rowspace-only, (¼,½) Codex, (½,0) MuP-A-only, (½,½) MuP. Each **lr-swept independently** (changing c moves the optimal η — a fixed-η comparison would confound c with lr).

**Prediction (to be tested, not assumed):** equal-ρ is verified-balanced at r≥256 and mildly A-heavy at r=64; the static analysis says the strong shape rules (Keller/Codex/MuP) over-correct into B-dominance, but only the *trained loss* settles whether any c≠1 arm beats baseline — that is what this measures. If a mild arm wins at r=64, extend to r=128/256 to test the predicted rank-decay.

Sweep: `params/e3_factor_scaling_llama32_openmath_r64.json` via `scripts/sweep/sweep_protagonist_factor_scaling.sh`.

In [ ]:
%load_ext autoreload
%autoreload 2

from lora_playground.plotting import compare_variants_figure

# label -> the (a,b) shape exponents (extra_where on top of common_where)
VARIANTS = {
    'c=1  (a0 b0)':   {'cw_factor_a': 0.0,  'cw_factor_b': 0.0},
    'Keller (a0 b½)': {'cw_factor_a': 0.0,  'cw_factor_b': 0.5},
    'A¼ only (a¼ b0)': {'cw_factor_a': 0.25, 'cw_factor_b': 0.0},
    'Codex (a¼ b½)':   {'cw_factor_a': 0.25, 'cw_factor_b': 0.5},
    'MuP-A (a½ b0)':   {'cw_factor_a': 0.5,  'cw_factor_b': 0.0},
    'MuP (a½ b½)':     {'cw_factor_a': 0.5,  'cw_factor_b': 0.5},
}

COMMON = {
    'optimizer': 'diag-shampoo-polar-lora',
    'model_name': 'meta-llama/Llama-3.2-1B',
    'lora_r': 64,
    'data_pipeline_version': 'packed_v1.1',
}

In [ ]:
# Final-loss vs lr (per-arm best lr) + best-lr trajectory; Δ vs c=1 in σ-units.
# allow_partial=True so in-flight runs render from the first eval.
fig = compare_variants_figure(
    VARIANTS,
    common_where=COMMON,
    ref_label='c=1  (a0 b0)',
    target_label='c=1  (a0 b0)',
    max_steps=9000,
    allow_partial=True,
    suptitle='Per-factor shape scaling — Llama-3.2-1B openmath r64 (protagonist)',
)

## Reading it

- **Left panel** = each arm's final eval-loss vs lr → read off each arm's *own* best lr (the lr-sweep-per-arm that avoids the c↔lr confound).
- **Right panel** = best-lr loss trajectory per arm.
- **Δ vs `c=1`** is reported in σ-units (AdamW noise floor). Decision rule: an arm only counts as beating baseline if its best-lr final loss is **> 1σ below** c=1. If no arm clears that, ship c=1 (the shipped optimizer) and the whole shape-scaling question closes negative.
- If a c≠1 arm wins here (r=64, max imbalance), re-point COMMON to r=128 then r=256 to test whether the win **decays with rank** (the prediction). A win that vanishes by r=256 means a rank-gated rule, not a static one.